In [1]:
from vllm import LLM, SamplingParams
import pandas as pd

train = r'/home/flopezp/Kurosagol/FOLIO/FOLIO/folio_test.jsonl'
dataset = pd.read_json(train, lines = True)

trans_prompt = """
    Given a set of premises, the task is to parse the problem and the question into first-order logic formulars. Answer only with the translated premises.
    The grammar of the first-order logic formular is defined as follows:
    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2
    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2
    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2
    4) logical negation of expr1: ¬expr1
    5) expr1 implies expr2: expr1 → expr2
    6) expr1 if and only if expr2: expr1 ↔ expr2
    7) logical universal quantification: ∀x
    8) logical existential quantification: ∃x
    --------------
    Natural Language Premises:
    All people who regularly drink coffee are dependent on caffeine. People either regularly drink coffee or joke about being addicted to caffeine. No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    Predicates:
    Dependent(x) ::: x is a person dependent on caffeine.
    Drinks(x) ::: x regularly drinks coffee.
    Jokes(x) ::: x jokes about being addicted to caffeine.
    Unaware(x) ::: x is unaware that caffeine is a drug.
    Student(x) ::: x is a student.
    FOL Premises:
    ∀x (Drinks(x) → Dependent(x)) ::: All people who regularly drink coffee are dependent on caffeine.
    ∀x (Drinks(x) ⊕ Jokes(x)) ::: People either regularly drink coffee or joke about being addicted to caffeine.
    ∀x (Jokes(x) → ¬Unaware(x)) ::: No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. 
    (Student(rina) ∧ Unaware(rina)) ⊕ ¬(Student(rina) ∨ Unaware(rina)) ::: Rina is either a student and unaware that caffeine is a drug, or neither a student nor unaware that caffeine is a drug. 
    ¬(Dependent(rina) ∧ Student(rina)) → (Dependent(rina) ∧ Student(rina)) ⊕ ¬(Dependent(rina) ∨ Student(rina)) ::: If Rina is not a person dependent on caffeine and a student, then Rina is either a person dependent on caffeine and a student, or neither a person dependent on caffeine nor a student.
    --------------
    
    Natural Language Premises:
    {}
    FOL Premises:
"""

prompts = [trans_prompt.format(dataset['premises'][i]) for i in range(len(dataset['premises']))]
print(len(prompts))

sampling_params = SamplingParams(temperature = 0.2, top_p = 0.9, max_tokens = 4000, seed = 67, logprobs = 10)
llm = LLM(model = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-14B', max_model_len = 2000, quantization = 'fp8', enforce_eager = True)
outputs = llm.generate(prompts, sampling_params)
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text.strip('\n')
    print(f"Prompt:    {prompt!r}")
    print(f"Output:    {generated_text!r}")
    print("-" * 60)

226
INFO 03-24 16:11:50 [utils.py:233] non-default args: {'max_model_len': 2000, 'disable_log_stats': True, 'quantization': 'fp8', 'enforce_eager': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-14B'}
INFO 03-24 16:13:11 [model.py:533] Resolved architecture: Qwen2ForCausalLM
INFO 03-24 16:13:11 [model.py:1582] Using max model len 2000
INFO 03-24 16:13:11 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-24 16:13:12 [vllm.py:754] Asynchronous scheduling is enabled.
WARNING 03-24 16:13:12 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-24 16:13:12 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-24 16:13:12 [vllm.py:964] Cudagraph is disabled under eager mode
INFO 03-24 16:13:12 [compilation.py:289] Enabled custom fusions: norm_qu

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore pid=159227) INFO 03-24 16:14:37 [default_loader.py:384] Loading weights took 3.48 seconds
(EngineCore pid=159227) INFO 03-24 16:14:38 [gpu_model_runner.py:4566] Model loading took 15.24 GiB memory and 84.352272 seconds
(EngineCore pid=159227) INFO 03-24 16:14:40 [gpu_worker.py:456] Available KV cache memory: 11.88 GiB
(EngineCore pid=159227) INFO 03-24 16:14:40 [kv_cache_utils.py:1316] GPU KV cache size: 64,864 tokens
(EngineCore pid=159227) INFO 03-24 16:14:40 [kv_cache_utils.py:1321] Maximum concurrency for 2,000 tokens per request: 32.43x
(EngineCore pid=159227) INFO 03-24 16:14:40 [core.py:281] init engine (profile, create kv cache, warmup model) took 2.15 seconds
(EngineCore pid=159227) WARNING 03-24 16:14:41 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=159227) WARNING 03-24 16:14:41 [vllm.py:799] Inductor compilation was disabled by user settings, optimizati

Rendering prompts:   0%|          | 0/226 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/226 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt:    '\n    Given a set of premises, the task is to parse the problem and the question into first-order logic formulars. Answer only with the translated premises.\n    The grammar of the first-order logic formular is defined as follows:\n    1) logical conjunction of expr1 and expr2: expr1 ∧ expr2\n    2) logical disjunction of expr1 and expr2: expr1 ∨ expr2\n    3) logical exclusive disjunction of expr1 and expr2: expr1 ⊕ expr2\n    4) logical negation of expr1: ¬expr1\n    5) expr1 implies expr2: expr1 → expr2\n    6) expr1 if and only if expr2: expr1 ↔ expr2\n    7) logical universal quantification: ∀x\n    8) logical existential quantification: ∃x\n    --------------\n    Natural Language Premises:\n    All people who regularly drink coffee are dependent on caffeine. People either regularly drink coffee or joke about being addicted to caffeine. No one who jokes about being addicted to caffeine is unaware that caffeine is a drug. Rina is either a student and unaware that caffe

In [13]:
import re

texto = outputs[0].outputs[0].text

end_think = re.search('</think>', texto)
text_chido = texto[end_think.end():].strip('\n')
text_chido.split('\n')

['    FOL Premises:',
 '    ∀x (ProfessionalSoccerPlayer(x) → ¬ProfessionalBasketballPlayer(x)) ::: No professional soccer players play professional basketball.',
 '    ∀x (ProfessionalSoccerDefender(x) → ProfessionalSoccerPlayer(x)) ::: All professional soccer defenders are professional soccer players.',
 '    ∀x (ProfessionalCenterback(x) → ProfessionalSoccerDefender(x)) ::: All professional centerbacks are professional soccer defenders.',
 '    ∃x (Athlete(x) ∧ ProfessionalCenterback(x)) ::: Some athletes are professional center-backs.',
 '    ProfessionalBasketballPlayer(curry) ::: Stephen Curry plays professional basketball.',
 '    --------------']

In [14]:
outputs[0].outputs[0].logprobs[0]

{262: Logprob(logprob=-0.05296507105231285, rank=1, decoded_token='   '),
 1066: Logprob(logprob=-3.3029651641845703, rank=2, decoded_token='    \n'),
 151649: Logprob(logprob=-5.05296516418457, rank=3, decoded_token='</think>'),
 14808: Logprob(logprob=-6.42796516418457, rank=4, decoded_token='    \n\n'),
 5501: Logprob(logprob=-7.05296516418457, rank=5, decoded_token='Please'),
 71486: Logprob(logprob=-7.67796516418457, rank=6, decoded_token='Alright'),
 5338: Logprob(logprob=-8.05296516418457, rank=7, decoded_token='First'),
 7771: Logprob(logprob=-8.11546516418457, rank=8, decoded_token='Your'),
 286: Logprob(logprob=-8.17796516418457, rank=9, decoded_token='       '),
 16: Logprob(logprob=-8.24046516418457, rank=10, decoded_token='1')}

In [ ]:
# logprobs = outputs[0].outputs[0].logprobs
# Corresponde a cada token generado. Obs: len(token) != len(text)
print(len(outputs[0].outputs[0].logprobs), len(outputs[0].outputs[0].token_ids), len(outputs[0].outputs[0].text))